## Objectives

This notebook identifies unusual sales patterns using statistical and machine learning techniques.

The following anomaly detection methods are implemented:

- Z-Score Analysis
- Interquartile Range (IQR)
- Isolation Forest

The detected anomalies will help identify unexpected demand spikes, stock shortages, promotional events, or operational issues.

In [ ]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [ ]:
from src.anomaly_detection import (
    detect_zscore,
    detect_iqr,
    detect_isolation_forest
)

In [ ]:
import os
from pathlib import Path
import pandas as pd

project_root = Path.cwd()  # assumes notebook is opened from project root

csv_path = project_root / "data" / "processed" / "clean_sales.csv"

df = pd.read_csv(
    csv_path,
    parse_dates=["date"],
    index_col="date"
)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(df.index, df["sales"])

plt.title("Daily Sales")

plt.xlabel("Date")

plt.ylabel("Sales")

plt.show()

In [ ]:
df = detect_zscore(df)

In [ ]:
df[["sales","z_score","z_anomaly"]].head()

In [ ]:
print("Total Z-Score Anomalies:", df["z_anomaly"].sum())

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    df.index,
    df["sales"],
    label="Sales"
)

plt.scatter(
    df[df["z_anomaly"]].index,
    df[df["z_anomaly"]]["sales"],
    color="red",
    s=60,
    label="Anomaly"
)

plt.legend(loc="upper left")

plt.title("Z-Score Anomaly Detection")

plt.show()

In [ ]:
import gc

df.drop(
    columns=["z_score"],
    inplace=True,
    errors="ignore"
)

gc.collect()

In [ ]:
df = detect_iqr(df)

In [ ]:
df[["sales","iqr_anomaly"]].head()

In [ ]:
print("Total IQR Anomalies:", df["iqr_anomaly"].sum())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

# Plot every 100th point for normal sales
plot_df = df.iloc[::100]

plt.plot(
    plot_df.index,
    plot_df["sales"],
    label="Sales"
)

# Plot all anomalies
anomaly_mask = df["iqr_anomaly"].values

plt.scatter(
    df.index[anomaly_mask],
    df["sales"].values[anomaly_mask],
    color="orange",
    s=60,
    label="Anomaly"
)

plt.legend(loc="upper left")

plt.title("IQR Anomaly Detection")

plt.show()

In [ ]:
import gc
import matplotlib.pyplot as plt

plt.close("all")

# Delete temporary variables if they exist
for var in ["plot_df", "anomalies"]:
    if var in globals():
        del globals()[var]

gc.collect()

In [ ]:
df = detect_isolation_forest(df)

In [ ]:
import gc
import matplotlib.pyplot as plt

plt.close("all")
gc.collect()

In [ ]:
df.drop(
    columns=["z_score"], 
    inplace=True, 
    errors="ignore"
)

gc.collect()

In [ ]:
df[["sales","iforest","iforest_anomaly"]].head()

In [ ]:
print("Total Isolation Forest Anomalies:", df["iforest_anomaly"].sum())

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    df.index,
    df["sales"],
    label="Sales"
)

plt.scatter(
    df[df["iforest_anomaly"]].index,
    df[df["iforest_anomaly"]]["sales"],
    color="green",
    s=60,
    label="Anomaly"
)

plt.legend()

plt.title("Isolation Forest Anomaly Detection")

plt.show()

In [ ]:
comparison = pd.DataFrame({

    "Method":[
        "Z-Score",
        "IQR",
        "Isolation Forest"
    ],

    "Detected Anomalies":[

        df["z_anomaly"].sum(),

        df["iqr_anomaly"].sum(),

        df["iforest_anomaly"].sum()

    ]

})

comparison

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(

    comparison["Method"],

    comparison["Detected Anomalies"]

)

plt.title("Comparison of Anomaly Detection Methods")

plt.ylabel("Number of Anomalies")

plt.show()

In [ ]:
df[
    df["iforest_anomaly"]
].head(10)

In [ ]:
from pathlib import Path

project_root = Path(r"C:\Users\Jainam\OneDrive\Documents\Infotact Solutions\Project 2")

output_path = project_root / "data" / "processed" / "anomaly_detection.csv"

df.to_csv(output_path, index=False)

# Conclusion

Three anomaly detection techniques were successfully applied to the cleaned supply chain dataset.

### Key Findings

- **Z-Score** identified observations that deviated significantly from the mean.
- **IQR** detected extreme values using quartile-based statistics.
- **Isolation Forest** identified complex anomalies using an unsupervised machine learning algorithm.

### Business Value

The detected anomalies may indicate:

- Unexpected spikes in product demand
- Inventory shortages
- Promotional campaigns
- Seasonal events
- Data quality issues
- Operational disruptions

These insights help supply chain managers investigate unusual events and improve inventory planning.